1. Import Libraries and Load the 3 datasets

In [ ]:
import pandas as pd
import numpy as np

# Load datasets while preserving IDs as text
charging = pd.read_csv("charging_sessions.csv.gz", dtype={
    "session_id": "string",
    "vehicle_id": "string",
    "station_id": "string"
})

vehicles = pd.read_csv("vehicles.csv", dtype={
    "vehicle_id": "string",
    "home_zip": "string"
})

stations = pd.read_csv("stations.csv", dtype={
    "station_id": "string",
    "zip": "string"
})

print("Datasets loaded successfully!")

2. Record Original row counts

In [ ]:
original_counts = {
    "Charging Sessions": len(charging),
    "Vehicles": len(vehicles),
    "Public Stations": len(stations)
}

for name, count in original_counts.items():
    print(f"{name}: {count:,} rows")

3. Inspect Datasets 

In [ ]:
for name, df in {
    "Charging Sessions": charging,
    "Vehicles": vehicles,
    "Public Stations": stations
}.items():

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print("Shape:", df.shape)

    print("\nMissing values:")
    print(df.isnull().sum())

    print("\nDuplicate rows:", df.duplicated().sum())

    print("\nData types:")
    print(df.dtypes)

4. Save initial copy of row counts

In [ ]:
# Record original row counts
original_row_counts = {
    name: len(df)
    for name, df in datasets.items()
}

print("Original row counts:")
for name, count in original_row_counts.items():
    print(f"{name}: {count:,} rows")

4. Create cleaning log

In [ ]:
cleaning_log = []

def log_cleaning(dataset, step, before, after, reason):
    cleaning_log.append({
        "Dataset": dataset,
        "Cleaning Step": step,
        "Rows Before": before,
        "Rows After": after,
        "Rows Removed": before - after,
        "Reason": reason
    })

def show_cleaning_log():
    return pd.DataFrame(cleaning_log)
def show_cleaning_log():
    return pd.DataFrame(cleaning_log)

5. Clean Vehicle Dataset

In [ ]:
vehicles_clean = vehicles.copy()

# Remove extra spaces and standardize IDs
vehicles_clean["vehicle_id"] = vehicles_clean["vehicle_id"].str.strip().str.upper()

# Remove records without vehicle IDs
before = len(vehicles_clean)
vehicles_clean = vehicles_clean.dropna(subset=["vehicle_id"])
log_cleaning(
    "Vehicles",
    "Remove missing vehicle IDs",
    before,
    len(vehicles_clean),
    "A vehicle must have an ID to connect it to charging sessions."
)

# Remove duplicate vehicle IDs
before = len(vehicles_clean)
vehicles_clean = vehicles_clean.drop_duplicates(
    subset=["vehicle_id"],
    keep="first"
)
log_cleaning(
    "Vehicles",
    "Remove duplicate vehicle IDs",
    before,
    len(vehicles_clean),
    "Each vehicle should appear only once."
)

# Convert numerical columns
for column in ["battery_kwh", "rated_kwh_per_100mi", "replaced_gas_car_mpg"]:
    vehicles_clean[column] = pd.to_numeric(
        vehicles_clean[column], errors="coerce"
    )

# Convert registration month
vehicles_clean["registration_month"] = pd.to_datetime(
    vehicles_clean["registration_month"],
    format="%Y-%m",
    errors="coerce"
)

print("Clean vehicle records:", len(vehicles_clean))
print("Unique vehicle IDs:", vehicles_clean["vehicle_id"].nunique())

6. Clean Station Dataset

In [ ]:
stations_clean = stations.copy()

# Standardize station IDs
stations_clean["station_id"] = (
    stations_clean["station_id"].str.strip().str.upper()
)

# Remove missing station IDs
before = len(stations_clean)
stations_clean = stations_clean.dropna(subset=["station_id"])

log_cleaning(
    "Stations",
    "Remove missing station IDs",
    before,
    len(stations_clean),
    "A public station must have an ID to match charging sessions."
)

# Remove duplicate station IDs
before = len(stations_clean)
stations_clean = stations_clean.drop_duplicates(
    subset=["station_id"],
    keep="first"
)

log_cleaning(
    "Stations",
    "Remove duplicate station IDs",
    before,
    len(stations_clean),
    "Each public charging station should appear once."
)

# Convert station opening date
stations_clean["open_date"] = pd.to_datetime(
    stations_clean["open_date"],
    errors="coerce"
)

print("Clean public stations:", len(stations_clean))
print("Unique station IDs:", stations_clean["station_id"].nunique())

7. Clean Charging Sessions

In [ ]:
charging_clean = charging.copy()

# Standardize text columns
text_columns = [
    "session_id",
    "vehicle_id",
    "station_id",
    "port_type",
    "connector",
    "home_or_public"
]

for column in text_columns:
    charging_clean[column] = charging_clean[column].str.strip()

# Standardize identifiers
charging_clean["session_id"] = charging_clean["session_id"].str.upper()
charging_clean["vehicle_id"] = charging_clean["vehicle_id"].str.upper()
charging_clean["station_id"] = charging_clean["station_id"].str.upper()

# Standardize charging location
charging_clean["home_or_public"] = (
    charging_clean["home_or_public"].str.lower()
)

# Standardize charger types
charging_clean["port_type"] = (
    charging_clean["port_type"].str.lower()
)

charging_clean["connector"] = (
    charging_clean["connector"].str.upper()
)

# Convert kWh to numeric
charging_clean["kwh"] = pd.to_numeric(
    charging_clean["kwh"], errors="coerce"
)

# Convert timestamps to datetime
charging_clean["start_time"] = pd.to_datetime(
    charging_clean["start_time"],
    format="%Y-%m-%d %H:%M",
    errors="coerce"
)

charging_clean["end_time"] = pd.to_datetime(
    charging_clean["end_time"],
    format="%Y-%m-%d %H:%M",
    errors="coerce"
)

print("Charging sessions inspected:", len(charging_clean))

Remove missing essential information

In [ ]:
required_columns = [
    "session_id",
    "vehicle_id",
    "station_id",
    "start_time",
    "end_time",
    "kwh",
    "home_or_public"
]

before = len(charging_clean)

charging_clean = charging_clean.dropna(
    subset=required_columns
)

log_cleaning(
    "Charging Sessions",
    "Remove missing essential values",
    before,
    len(charging_clean),
    "Sessions need identifiers, timestamps, electricity consumption, and charging location."
)

Remove duplicate sessions

In [ ]:
before = len(charging_clean)

charging_clean = charging_clean.drop_duplicates(
    subset=["session_id"],
    keep="first"
)

log_cleaning(
    "Charging Sessions",
    "Remove duplicate session IDs",
    before,
    len(charging_clean),
    "Each charging session should appear only once."
)

Remove invalid charging values

In [ ]:
before = len(charging_clean)

charging_clean = charging_clean[
    (charging_clean["kwh"] > 0) &
    (charging_clean["end_time"] > charging_clean["start_time"]) &
    (charging_clean["home_or_public"].isin(["home", "public"]))
].copy()

log_cleaning(
    "Charging Sessions",
    "Remove invalid charging records",
    before,
    len(charging_clean),
    "Charging sessions must have positive energy, a valid duration, and a recognized location."
)

print("Remaining charging sessions:", len(charging_clean))

8. Match Charging Sessions to Vehicles

In [ ]:
# Identify sessions with no matching vehicle
vehicle_ids = set(vehicles_clean["vehicle_id"])

charging_clean["vehicle_match"] = (
    charging_clean["vehicle_id"].isin(vehicle_ids)
)

unmatched_vehicles = charging_clean[
    ~charging_clean["vehicle_match"]
].copy()

print("Sessions with matching vehicles:",
      charging_clean["vehicle_match"].sum())

print("Sessions without matching vehicles:",
      len(unmatched_vehicles))

Save unmatched records for documentation

In [ ]:
unmatched_vehicles.to_csv(
    "unmatched_vehicle_sessions.csv",
    index=False
)

Remove unmatched records from analysis dataset

In [ ]:
before = len(charging_clean)

charging_clean = charging_clean[
    charging_clean["vehicle_match"]
].copy()

charging_clean = charging_clean.drop(columns=["vehicle_match"])

log_cleaning(
    "Charging Sessions",
    "Remove sessions without matching vehicles",
    before,
    len(charging_clean),
    "Every analyzed charging session must belong to one of the 10,000 mandate vehicles."
)

9. Match Charging Sessions to Stations
    There are two different rules:
- Public: The station_id must exist in stations.csv.
- Home: The station_id must equal H- followed by the matching vehicle_id.

In [ ]:
public_station_ids = set(stations_clean["station_id"])

# Check public station matches
charging_clean["station_match"] = np.where(
    charging_clean["home_or_public"] == "public",
    charging_clean["station_id"].isin(public_station_ids),

    # Home station must match its vehicle ID
    charging_clean["station_id"] ==
    "H-" + charging_clean["vehicle_id"]
)

# Identify unmatched stations
unmatched_stations = charging_clean[
    ~charging_clean["station_match"]
].copy()

print("Sessions with matching stations:",
      charging_clean["station_match"].sum())

print("Sessions without matching stations:",
      len(unmatched_stations))

Save unmatchhed station records

In [ ]:
unmatched_stations.to_csv(
    "unmatched_station_sessions.csv",
    index=False
)

Remove any records with invalid station relationships

In [ ]:
before = len(charging_clean)

charging_clean = charging_clean[
    charging_clean["station_match"]
].copy()

charging_clean = charging_clean.drop(
    columns=["station_match"]
)

log_cleaning(
    "Charging Sessions",
    "Remove sessions without matching stations",
    before,
    len(charging_clean),
    "Public sessions must match stations.csv; home sessions must match their vehicle's H- station ID."
)

10. Check charging sessions against battery capacity
a session cannot deliver more electricity than the vehicle's usable battery capacity.

We can check this by connecting the vehicle information to each session.

In [ ]:
charging_clean = charging_clean.merge(
    vehicles_clean[["vehicle_id", "battery_kwh"]],
    on="vehicle_id",
    how="left",
    validate="many_to_one"
)

# Identify sessions exceeding battery capacity
battery_issues = charging_clean[
    charging_clean["kwh"] > charging_clean["battery_kwh"]
].copy()

print("Sessions exceeding battery capacity:", len(battery_issues))

Save records for inspection, remove after if neccissary

In [ ]:
battery_issues.to_csv(
    "battery_capacity_issues.csv",
    index=False
)

11. Seperate home and public charging sessions

In [ ]:
# Separate home charging
home_charging = charging_clean[
    charging_clean["home_or_public"] == "home"
].copy()

# Separate public charging
public_charging = charging_clean[
    charging_clean["home_or_public"] == "public"
].copy()

print("Home charging sessions:", len(home_charging))
print("Public charging sessions:", len(public_charging))

print("Total separated sessions:",
      len(home_charging) + len(public_charging))

print("Total cleaned sessions:", len(charging_clean))

12. Add vehicle and station information

In [ ]:
# Add vehicle information to home sessions
home_charging = home_charging.merge(
    vehicles_clean,
    on="vehicle_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_vehicle")
)

# Add vehicle and public station information
public_charging = public_charging.merge(
    vehicles_clean,
    on="vehicle_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_vehicle")
)

public_charging = public_charging.merge(
    stations_clean,
    on="station_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_station")
)

print("Home charging rows:", len(home_charging))
print("Public charging rows:", len(public_charging))

13. Save cleaned datasets

In [ ]:
charging_clean.to_csv("charging_sessions_clean.csv", index=False)

home_charging.to_csv("home_charging_clean.csv", index=False)

public_charging.to_csv("public_charging_clean.csv", index=False)

vehicles_clean.to_csv("vehicles_clean.csv", index=False)

stations_clean.to_csv("stations_clean.csv", index=False)

print("All cleaned datasets saved!")

14. Show cleaning log

In [ ]:
cleaning_report = show_cleaning_log()

display(cleaning_report)